# EDA — Customer Support Tickets (LogiStore)

Dataset Kaggle : `customer_support_tickets_200k.csv` (200 000 lignes, 30 colonnes).

**But de l'EDA** : valider la pertinence du dataset pour le projet RAG (couverture, unicité, longueur des textes), et calibrer les validateurs Pydantic du modèle `Ticket`.

**Findings principaux** (résumé exécutif) :
1. Dataset très synthétique : seulement **10 issues uniques × 10 resolutions uniques = 100 paires textuelles distinctes**.
2. Le champ `issue_complexity_score` va de **1 à 10**, pas 1 à 5 — validateur corrigé en conséquence.
3. Textes très courts (55–80 caractères) — pas besoin de chunking.
4. Multilingue (6 langues équilibrées : EN, FR, ES, DE, JP, ZH).
5. Pour obtenir un index riche, la **clé de dédup** combine `(category, product, language, issue, resolution)` → **57 884 docs uniques** sur 200K (29%).


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
CSV = Path("../data/raw/customer_support_tickets.csv")
df = pd.read_csv(CSV)
print(df.shape)
df.head()

## 1. Schéma & valeurs manquantes

In [ ]:
df.info()

In [ ]:
nulls = df.isna().sum()
nulls[nulls > 0]

Seule `browser` a des NaN (~20%). Aucun champ critique pour le RAG n'est lacunaire.

## 2. Distributions des champs numériques

Crucial pour calibrer les bornes des validateurs Pydantic.

In [ ]:
num_cols = df.select_dtypes(include="number").columns
df[num_cols].describe().T[["min", "max", "mean", "std"]].round(2)

**Observation clé** : `issue_complexity_score` va de **1 à 10**, alors qu'on supposait initialement 1 à 5. Le validateur Pydantic `Ticket.issue_complexity_score` a été ajusté à `ge=1, le=10`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df["issue_complexity_score"].value_counts().sort_index().plot.bar(ax=ax)
ax.set_title("Distribution de issue_complexity_score (1..10)")
ax.set_xlabel("score"); ax.set_ylabel("tickets")
plt.show()

## 3. Distributions catégorielles

Les 10 catégories de tickets et les 10 produits sont presque parfaitement équilibrés : c'est un dataset synthétique uniforme.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df["category"].value_counts().plot.barh(ax=axes[0], title="Catégories")
df["product"].value_counts().plot.barh(ax=axes[1], title="Produits")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["language"].value_counts().plot.bar(ax=axes[0], title="Langues")
df["region"].value_counts().plot.bar(ax=axes[1], title="Régions")
plt.tight_layout(); plt.show()

## 4. Longueur des textes

Les champs `issue_description` (55–79 chars) et `resolution_notes` (60–80 chars) sont **très courts**. Un ticket entier tient largement dans la fenêtre de 8K tokens de BGE-M3.

**Conséquence pour le RAG** : pas besoin de chunking. Stratégie d'indexation = **1 document par ticket**, content = `[CATEGORY] ... [PRODUCT] ... [LANGUAGE] ... [REGION] ... [ISSUE] ... [RESOLUTION] ...`.

In [ ]:
lens_iss = df["issue_description"].fillna("").astype(str).str.len()
lens_res = df["resolution_notes"].fillna("").astype(str).str.len()

fig, ax = plt.subplots(figsize=(10, 4))
lens_iss.plot.hist(bins=60, ax=ax, alpha=0.6, label="issue_description")
lens_res.plot.hist(bins=60, ax=ax, alpha=0.6, label="resolution_notes")
ax.set_xlabel("Longueur (caractères)"); ax.legend(); ax.set_title("Longueur des champs textuels")
plt.show()

## 5. Unicité textuelle — le finding principal

C'est l'observation qui a déterminé la stratégie de dédup.

In [ ]:
for col in ["issue_description", "resolution_notes", "category", "product", "language"]:
    print(f"  {col:25s} -> {df[col].nunique()} valeurs uniques")

In [ ]:
# Différentes stratégies de clé de dédup -- combien de docs uniques obtient-on ?
for cols in [
    ["issue_description", "resolution_notes"],
    ["category", "product", "language", "issue_description", "resolution_notes"],
    ["category", "product", "language", "region", "issue_description", "resolution_notes"],
    ["category", "product", "language", "region", "priority", "issue_description", "resolution_notes"],
]:
    key = df[cols].astype(str).agg("|".join, axis=1)
    print(f"  {'+'.join(cols):90s} -> {key.nunique():>7,} uniques ({100*key.nunique()/len(df):.1f}%)")

**Décision retenue** : clé `(category, product, language, issue, resolution)` → 57 884 docs uniques (29%).  
Le choix est documenté dans `src/ragtime/ingest/chunker.py::compute_document_id`.

In [ ]:
# Top 10 paires (issue, resolution) les plus fréquentes
df["_pair"] = df["issue_description"].astype(str).str[:50] + " || " + df["resolution_notes"].astype(str).str[:50]
df["_pair"].value_counts().head(10)

## 6. Dates & temporalité

In [ ]:
df["_created"] = pd.to_datetime(df["ticket_created_date"], errors="coerce")
df["_resolved"] = pd.to_datetime(df["ticket_resolved_date"], errors="coerce")
print(f"Créés     : {df['_created'].min()} -> {df['_created'].max()}")
print(f"Résolus   : {df['_resolved'].min()} -> {df['_resolved'].max()}")
df["_created"].dt.to_period("M").value_counts().sort_index().plot.line(figsize=(12, 4), title="Tickets créés par mois")
plt.show()

## 7. Conclusions opérationnelles

| Aspect | Constat | Décision |
|---|---|---|
| Volume utile | 100 paires textuelles uniques (cartésien 10×10) | Enrichir la clé de dédup avec category + product + language |
| Validateurs Pydantic | `issue_complexity_score` réellement dans 1..10 | Borne ajustée dans `models.py` |
| Chunking | Textes 55-80 chars | 1 doc/ticket, pas de split |
| Multilinguisme | 6 langues équilibrées | BGE-M3 (multilingue) confirmé |
| PII | `customer_name`, `customer_email` présents | Dropés à l'étape `normalizer` (extra='ignore') |
| Sample MVP | 10 000 lignes → ~9 200 docs uniques (92%) | `DATASET_SAMPLE_SIZE=10000` (défaut) |
